# Imports + paths + load data

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

import joblib


In [2]:
# --- Paths ---
cwd = Path.cwd()
if cwd.name == "notebooks":
    project_root = cwd.parent
elif (cwd / "notebooks").exists():
    project_root = cwd
else:
    project_root = cwd

data_path = project_root / "data" / "processed" / "original_data.csv"

final_dir = project_root / "data" / "final"
final_dir.mkdir(parents=True, exist_ok=True)

models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# --- Load ---
df = pd.read_csv(data_path)
print("Loaded:", data_path.resolve())
print("Shape:", df.shape)

# Explicit target
target_col = "Diabetes_binary"
assert target_col in df.columns, f"{target_col} not found!"

df.head()

Loaded: C:\Users\Lu\OneDrive\ToU\chl_Classification\data\processed\original_data.csv
Shape: (253680, 22)


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,1,1,1,40,1,0,0,0,0,1,...,0,5,18,15,1,0,9,4,3,0
1,0,0,0,25,1,0,0,1,0,0,...,1,3,0,0,0,0,7,6,1,0
2,1,1,1,28,0,0,0,0,1,0,...,1,5,30,30,1,0,9,4,8,0
3,1,0,1,27,0,0,0,1,1,1,...,0,2,0,0,0,0,11,3,6,0
4,1,1,1,24,0,0,0,1,1,1,...,0,2,3,0,0,0,11,5,4,0


# Train/test split (Stratified because of class imbalance)

Take the selected feature / candidate features into explicit lists to create reproducability (not dynamic cutoffs >5)

In [3]:
# --- EDA-driven feature selection (explicit & reproducible) ---

core_features = [
    "HighBP", "HighChol", "HeartDiseaseorAttack", "Stroke", "DiffWalk",
    "GenHlth", "BMI", "Age", "PhysHlth", "MentHlth",
    "Sex", "Education", "Income",
]

candidate_features = [
    "Smoker", "PhysActivity", "Fruits", "Veggies", "HvyAlcoholConsump",
    "CholCheck", "AnyHealthcare", "NoDocbcCost",
]

selected_features = core_features + candidate_features

# Sanity checks
missing = [c for c in selected_features if c not in df.columns]
assert len(missing) == 0, f"Selected features missing in df: {missing}"
assert target_col in df.columns, f"Target column missing: {target_col}"

X = df[selected_features].copy()
y = df[target_col].copy()

print(f"Selected {len(selected_features)} features for modeling.")
print("X shape:", X.shape)
print("y shape:", y.shape)

# Class balance overview
counts = y.value_counts()
props = y.value_counts(normalize=True)
display(counts)
display(props.rename("proportion"))


Selected 21 features for modeling.
X shape: (253680, 21)
y shape: (253680,)


Diabetes_binary
0    218334
1     35346
Name: count, dtype: int64

Diabetes_binary
0    0.860667
1    0.139333
Name: proportion, dtype: float64

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

# sanity check: stratification preserved
display(y_train.value_counts(normalize=True).rename("train_proportion"))
display(y_test.value_counts(normalize=True).rename("test_proportion"))

# Save CSVs
X_train.to_csv(final_dir / "X_train.csv", index=False)
X_test.to_csv(final_dir / "X_test.csv", index=False)
y_train.to_csv(final_dir / "y_train.csv", index=False)
y_test.to_csv(final_dir / "y_test.csv", index=False)

print("Saved to:", final_dir.resolve())


Train shape: (202944, 21) Test shape: (50736, 21)


Diabetes_binary
0    0.860666
1    0.139334
Name: train_proportion, dtype: float64

Diabetes_binary
0    0.860671
1    0.139329
Name: test_proportion, dtype: float64

Saved to: C:\Users\Lu\OneDrive\ToU\chl_Classification\data\final


# Build preprocessing pipeline

Define feature groups for preprocessing

Instead of auto-detecting *discrete vs continuous* by a `nunique` cutoff, we explicitly group features as:
- **binary (0/1)**: impute only (no scaling needed)
- **ordinal** (ordered categories like GenHlth/Age/Education/Income): impute + scale (helps Logistic Regression)
- **numeric** (BMI/PhysHlth/MentHlth): impute + scale

This matches the EDA-driven selection and avoids misclassifying ordinal variables.

In [5]:
# --- Feature groups for preprocessing (aligned with EDA + model needs) ---

binary_cols = [
    "HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost",
    "DiffWalk", "Sex",
]

# These are coded as ordered categories (ordinal) in this dataset
ordinal_cols = ["GenHlth", "Age", "Education", "Income"]

# Continuous-ish / count-like
numeric_cols = ["BMI", "PhysHlth", "MentHlth"]

# Ensure the grouping matches what we actually selected
grouped = set(binary_cols + ordinal_cols + numeric_cols)
selected = set(X.columns)

missing_in_groups = sorted(selected - grouped)
extra_in_groups = sorted(grouped - selected)

assert len(missing_in_groups) == 0, f"These selected features are not assigned to a group: {missing_in_groups}"
assert len(extra_in_groups) == 0, f"These grouped features are not in selected_features: {extra_in_groups}"

print("Feature groups:")
print(f"- binary_cols:  {len(binary_cols)}")
print(f"- ordinal_cols: {len(ordinal_cols)}")
print(f"- numeric_cols: {len(numeric_cols)}")

Feature groups:
- binary_cols:  14
- ordinal_cols: 4
- numeric_cols: 3


Pipeline for continuous-ish: impute then scale

In [6]:
# --- Pipelines ---

# Numeric / count-like: impute + scale (important for Logistic Regression)
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Ordinal: treat as numeric order + scale (important for Logistic Regression)
ordinal_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Binary: impute most frequent; no scaling needed
binary_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_cols),
        ("ord", ordinal_pipe, ordinal_cols),
        ("bin", binary_pipe, binary_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

# Fit on TRAIN only
preprocessor.fit(X_train)

print("Preprocessor fitted on X_train.")


Preprocessor fitted on X_train.


Transform train/test

In [7]:
X_train_t = preprocessor.transform(X_train)
X_test_t = preprocessor.transform(X_test)

print("Transformed train shape:", X_train_t.shape)
print("Transformed test shape:", X_test_t.shape)

Transformed train shape: (202944, 21)
Transformed test shape: (50736, 21)


# Save preprocessor

In [8]:
preprocessor_path = models_dir / "preprocessor.joblib"
joblib.dump(preprocessor, preprocessor_path)

print("Saved preprocessor:", preprocessor_path.resolve())


Saved preprocessor: C:\Users\Lu\OneDrive\ToU\chl_Classification\models\preprocessor.joblib
